# LangGraph Advanced

This companion notebook moves from LangGraph basics to reliable, realistic
agent systems. The original notebook remains unchanged. Each section starts
with learning goals, an easy explanation, a Mermaid topology, executable
code, assertions, and a small practice prompt.

## Learning path

1. Control agent autonomy and observe every streaming surface.
2. Make tools reliable with validation, retries, timeouts, and fallbacks.
3. Add typed outputs, durable checkpoints, human approval, and state boundaries.
4. Compose parallel and multi-agent systems with explicit runtime policy.

### How to study

Before running a section, predict its state transition and failure behavior.
After running it, read the assertion output and complete the practice prompt.

Model calls are real where model behavior is the subject. Small local tools and
data make reliability behavior deterministic. There is no mock provider or
silent fallback.

## Setup

Choose exactly one provider. The matching key must exist in the repository-root
.env. This is the same explicit provider contract used by the basics notebook.

In [ ]:
import asyncio
import operator
import os
import time
from concurrent.futures import ThreadPoolExecutor, TimeoutError as FutureTimeout
from pathlib import Path
import tempfile
from typing import Annotated, Literal, TypedDict

from dotenv import load_dotenv
from pydantic import BaseModel, Field, ValidationError
from langchain_core.messages import HumanMessage, SystemMessage
from langchain_core.runnables import RunnableConfig
from langchain_core.tools import tool
from langgraph.checkpoint.memory import InMemorySaver
from langgraph.graph import START, END, StateGraph
from langgraph.graph.message import add_messages
from langgraph.prebuilt import ToolNode, tools_condition
from langgraph.runtime import Runtime
from langgraph.types import Command, Send, interrupt

load_dotenv()
PROVIDER = "anthropic"  # or "openai"
ANTHROPIC_MODEL = "claude-sonnet-5"
OPENAI_MODEL = "gpt-4o"

def get_llm(model=None):
    if PROVIDER == "anthropic":
        key = os.environ.get("ANTHROPIC_API_KEY")
        if not key:
            raise RuntimeError("ANTHROPIC_API_KEY is missing for selected provider")
        from langchain_anthropic import ChatAnthropic
        return ChatAnthropic(model=model or ANTHROPIC_MODEL, api_key=key)
    if PROVIDER == "openai":
        key = os.environ.get("OPENAI_API_KEY")
        if not key:
            raise RuntimeError("OPENAI_API_KEY is missing for selected provider")
        from langchain_openai import ChatOpenAI
        return ChatOpenAI(model=model or OPENAI_MODEL, api_key=key)
    raise ValueError("PROVIDER must be anthropic or openai")

def get_text(message):
    content = message.content
    if isinstance(content, str):
        return content
    return "".join(block.get("text", "") for block in content if isinstance(block, dict))

llm = get_llm()
print("Provider:", PROVIDER, "| model:", getattr(llm, "model", getattr(llm, "model_name", "configured")))
print("Live provider check:", get_text(llm.invoke("Reply with exactly one word: ready")))

## 1. Explicit agents and ReAct

**Goal:** trace one bounded agent loop from request to tool call to final answer.


An LLM chain follows a fixed recipe: input -> prompt -> model -> output.
A workflow follows a graph whose routing we write. An agent repeatedly asks a
model which available action is useful, executes that action, observes it, and
decides again.

ReAct is the visible control loop:

~~~mermaid
flowchart LR
 U[User] --> A[Agent: choose next action]
 A -->|tool call| T[Tool executor]
 T -->|ToolMessage observation| A
 A -->|no tool call: final answer| E((END))
~~~

The message list is the agent trajectory: human request, AI tool call, tool
result, and the next AI response. The model does not execute Python; the graph
does. The agent stops when the model returns no tool call, but production
systems also enforce a step limit, time budget, permissions, and spend budget.

This notebook demonstrates both layers: max_steps is the intentional application-level stop, while recursion_limit remains LangGraph's hard safety net if routing logic misbehaves.
The word reason describes an action-selection phase, not exposed private
chain-of-thought.

Use a chain when the sequence is known. Use a workflow when deterministic
routing is clear. Use an agent when the next useful action depends on the
request and observations, while keeping policy decisions deterministic.

In [ ]:
@tool
def lookup_order(order_id: str) -> str:
    """Look up a demo order. Valid IDs include ORD-100 and ORD-200."""
    orders = {"ORD-100": "shipped; carrier=ParcelFox; expected=2026-07-28",
              "ORD-200": "processing; warehouse=Seoul; expected=2026-07-30"}
    return orders.get(order_id.upper(), "order %s was not found" % order_id)

@tool
def calculate_refund(amount: float, percentage: float) -> str:
    """Calculate a refund; percentage must be between 0 and 100."""
    if not 0 <= percentage <= 100:
        raise ValueError("percentage must be between 0 and 100")
    return "refund=%.2f" % (amount * percentage / 100)

agent_tools = [lookup_order, calculate_refund]
agent_llm = llm.bind_tools(agent_tools)

class AgentState(TypedDict):
    messages: Annotated[list, add_messages]
    steps: int
    max_steps: int

def agent_node(state: AgentState):
    response = agent_llm.invoke(state["messages"])
    return {"messages": [response], "steps": state["steps"] + 1}

agent_graph = StateGraph(AgentState)
agent_graph.add_node("agent", agent_node)
agent_graph.add_node("tools", ToolNode(agent_tools))
agent_graph.add_edge(START, "agent")
def route_after_agent(state):
    # Intentional application-level stop before the hard graph safety limit.
    if state["steps"] >= state["max_steps"]:
        return END
    return tools_condition(state)

agent_graph.add_conditional_edges("agent", route_after_agent, {"tools": "tools", END: END})
agent_graph.add_edge("tools", "agent")
agent_app = agent_graph.compile()

print(agent_app.get_graph().draw_mermaid())
result = agent_app.invoke(
    {"messages": [HumanMessage(content="Check ORD-100 and tell me its delivery status.")], "steps": 0, "max_steps": 4},
    config={"recursion_limit": 8},
)
for i, message in enumerate(result["messages"], 1):
    print(i, type(message).__name__, "|", getattr(message, "tool_calls", None) or get_text(message))
print("bounded agent steps:", result["steps"])

In [ ]:
# Agent autonomy is bounded by deterministic policy outside the ReAct loop.
class PolicyState(TypedDict):
    request: str
    decision: str

def policy_gate(state):
    amount = 900 if "900" in state["request"] else 100
    return {"decision": "human_review" if amount > 500 else "agent_allowed"}

policy = StateGraph(PolicyState)
policy.add_node("gate", policy_gate)
policy.add_node("agent_allowed", lambda s: {"decision": "agent ran within policy"})
policy.add_node("human_review", lambda s: {"decision": "escalated before agent execution"})
policy.add_edge(START, "gate")
policy.add_conditional_edges("gate", lambda s: s["decision"])
policy.add_edge("agent_allowed", END)
policy.add_edge("human_review", END)
print(policy.compile().invoke({"request": "Refund 900 dollars", "decision": ""})["decision"])

## 2. Complete streaming

**Goal:** choose the stream surface that matches a UI, audit log, or tracer.


Streaming changes how a completed graph is consumed, not its topology.

| Mode | Shows | Good for |
|---|---|---|
| values | complete state snapshots | teaching and debugging |
| updates | node state deltas | progress panels and audit logs |
| messages | model chunks plus metadata | chat and token UIs |
| custom | application progress events | long-running tools |
| astream_events | runnable lifecycle events | tracing and instrumentation |

~~~mermaid
flowchart LR
 I[Input] --> G[Graph]
 G --> V[values]
 G --> U[updates]
 G --> M[messages and tokens]
 G --> X[events]
~~~

Use token/messages streaming for perceived responsiveness, updates for node/tool
progress, custom events for domain progress, and async event streaming in a
WebSocket or async web handler. The code implements custom events and then
shows lifecycle events on a deterministic graph so provider message metadata
does not obscure the event API being taught.

In [ ]:
stream_app = agent_graph.compile()
stream_input = {"messages": [HumanMessage(content="What is the status of ORD-200?")], "steps": 0, "max_steps": 4}
print("--- updates ---")
for update in stream_app.stream(stream_input, stream_mode="updates"):
    print(update)
print("--- values ---")
for snapshot in stream_app.stream(stream_input, stream_mode="values"):
    print("messages:", len(snapshot["messages"]), "steps:", snapshot["steps"])

In [ ]:
print("--- direct token streaming ---")
for chunk in llm.stream("Explain LangGraph in one short sentence."):
    text = get_text(chunk)
    if text:
        print(text, end="", flush=True)
print()

message_state = TypedDict("MessageState", {"messages": Annotated[list, add_messages]})
message_graph = StateGraph(message_state)
message_graph.add_node("chat", lambda s: {"messages": [llm.invoke(s["messages"])]})
message_graph.add_edge(START, "chat")
message_graph.add_edge("chat", END)
message_app = message_graph.compile()
print("--- graph messages streaming (simple chat graph) ---")
for chunk, metadata in message_app.stream({"messages": [HumanMessage(content="Say hello in one short sentence.")]}, stream_mode="messages"):
    text = get_text(chunk)
    if text:
        print("[%s] %s" % (metadata.get("langgraph_node"), text), end="", flush=True)
print()

async def collect_lifecycle_events():
    class EventState(TypedDict):
        value: int
    event_graph = StateGraph(EventState)
    event_graph.add_node("double", lambda s: {"value": s["value"] * 2})
    event_graph.add_edge(START, "double")
    event_graph.add_edge("double", END)
    event_app = event_graph.compile()
    names = []
    async for event in event_app.astream_events({"value": 3}, version="v2"):
        if event["event"] not in names:
            names.append(event["event"])
    return names

lifecycle_events = await collect_lifecycle_events()
print("--- lifecycle event types ---")
print(lifecycle_events)
assert "on_chain_start" in lifecycle_events
assert "on_chain_end" in lifecycle_events
print("Lifecycle event assertions passed")

In [ ]:
from langgraph.config import get_stream_writer

class ProgressState(TypedDict):
    status: str

def progress_node(state: ProgressState):
    writer = get_stream_writer()
    for phase in ["validate", "contact_carrier", "parse_response"]:
        writer({"phase": phase, "status": "running"})
        time.sleep(0.01)
    return {"status": "delivered"}

progress_graph = StateGraph(ProgressState)
progress_graph.add_node("carrier_lookup", progress_node)
progress_graph.add_edge(START, "carrier_lookup")
progress_graph.add_edge("carrier_lookup", END)
print("--- custom stream events ---")
for event in progress_graph.compile().stream({"status": ""}, stream_mode="custom"):
    print(event)

## 3. Tool failure handling

**Goal:** distinguish validation errors, transient failures, timeouts, and fallback observations.


A tool boundary must handle invalid arguments, exceptions, transient failure,
timeouts, retry budgets, fallback semantics, and multiple calls. Do not hide
errors: turn them into explicit state or tool observations so the agent can
recover or escalate. The follow-up example places this boundary inside an
actual agent-to-tool loop rather than only calling a wrapper manually.

~~~mermaid
flowchart TD
 A[Tool call] --> V[Validate args]
 V -->|invalid| E[Error observation]
 V -->|valid| P[Primary tool]
 P -->|transient error| R{retry budget}
 R -->|yes| P
 R -->|no| F[Fallback or escalation]
 P --> O[Observation]
 F --> O
 O --> A
~~~

Retry transient errors, not invalid input. Put a timeout around every network
or subprocess boundary and label fallback data clearly.

In [ ]:
class ReliabilityState(TypedDict):
    query: str
    attempts: int
    result: str
    error: str
    source: str
    force_down: bool

primary_attempts = {"Seoul": 0}
def primary_weather(city):
    if city == "Seoul" and primary_attempts["Seoul"] < 2:
        primary_attempts["Seoul"] += 1
        raise ConnectionError("primary service temporarily unavailable")
    if city not in {"Seoul", "Busan"}:
        raise ValueError("city must be Seoul or Busan")
    return {"Seoul": "rain, 22C", "Busan": "cloudy, 25C"}[city]

def fallback_weather(city):
    return {"Seoul": "historical baseline: rain likely", "Busan": "historical baseline: cloudy"}.get(city, "no fallback data")

def call_with_timeout(fn, *args, timeout=0.25):
    with ThreadPoolExecutor(max_workers=1) as pool:
        future = pool.submit(fn, *args)
        try:
            return future.result(timeout=timeout)
        except FutureTimeout:
            future.cancel()
            raise TimeoutError("%s exceeded timeout" % fn.__name__)

def reliable_lookup(state):
    last_error = ""
    for attempt in range(1, 4):
        try:
            if state.get("force_down", False):
                raise ConnectionError("forced outage for fallback demonstration")
            return {"attempts": attempt, "result": call_with_timeout(primary_weather, state["query"]),
                    "error": "", "source": "primary"}
        except (ConnectionError, TimeoutError) as exc:
            last_error = str(exc)
            time.sleep(0.02 * attempt)
        except ValueError as exc:
            return {"attempts": attempt, "result": "", "error": str(exc), "source": "validation"}
    return {"attempts": 3, "result": fallback_weather(state["query"]), "error": last_error, "source": "fallback"}

reliability = StateGraph(ReliabilityState)
reliability.add_node("lookup", reliable_lookup)
reliability.add_edge(START, "lookup")
reliability.add_edge("lookup", END)
reliability_app = reliability.compile()
blank = {"query": "Seoul", "attempts": 0, "result": "", "error": "", "source": "", "force_down": False}
print("transient recovery:", reliability_app.invoke(blank))
print("invalid argument:", reliability_app.invoke({**blank, "query": "Mars"}))
forced_fallback = reliability_app.invoke({**blank, "force_down": True})
print("fallback after retry budget:", forced_fallback)
def slow_weather(city):
    time.sleep(0.05)
    return city
try:
    call_with_timeout(slow_weather, "Seoul", timeout=0.001)
except TimeoutError as exc:
    print("timeout path:", exc)
else:
    raise AssertionError("expected timeout was not raised")
assert forced_fallback["source"] == "fallback"
assert forced_fallback["attempts"] == 3
print("Failure-path assertions passed")

In [ ]:
def safe_refund(amount, percentage):
    try:
        return calculate_refund.invoke({"amount": amount, "percentage": percentage})
    except Exception as exc:
        return {"ok": False, "error_type": type(exc).__name__, "message": str(exc), "retryable": False}

print("invalid tool args:", safe_refund(100, 140))
print("multiple independent calls:", [safe_refund(100, 10), safe_refund(250, 20)])

In [ ]:
@tool
def resilient_weather(city: str) -> str:
    """Return weather with bounded retry and an explicit fallback observation."""
    try:
        return reliable_lookup({"query": city, "attempts": 0, "result": "", "error": "", "source": ""})["result"]
    except Exception as exc:
        return "TOOL_ERROR retryable=false message=%s" % exc

resilient_agent_llm = llm.bind_tools([resilient_weather])
class ResilientAgentState(TypedDict):
    messages: Annotated[list, add_messages]

def resilient_agent(state):
    return {"messages": [resilient_agent_llm.invoke(state["messages"])]}

resilient_graph = StateGraph(ResilientAgentState)
resilient_graph.add_node("agent", resilient_agent)
resilient_graph.add_node("tools", ToolNode([resilient_weather]))
resilient_graph.add_edge(START, "agent")
resilient_graph.add_conditional_edges("agent", tools_condition, {"tools": "tools", END: END})
resilient_graph.add_edge("tools", "agent")
resilient_result = resilient_graph.compile().invoke({
    "messages": [HumanMessage(content="What is the weather in Seoul?")]
})
print("agent-integrated resilient tool:", get_text(resilient_result["messages"][-1]))

## 4. Structured output and validation

**Goal:** make model output a typed contract and recover from invalid data.


Natural-language classification is brittle. with_structured_output turns the
model response into a typed contract. Pydantic validates shape and constraints;
semantic checks are still needed because a well-formed order ID may not exist. A production node should catch schema failures, retry with a repair instruction, and escalate after a bounded number of attempts; the example includes that recovery boundary.

~~~mermaid
flowchart LR
 I[Request] --> M[LLM and schema]
 M --> V{Pydantic validation}
 V -->|valid| R[Typed router]
 V -->|invalid| C[Retry, correct, or escalate]
~~~

Use this for routing decisions, extraction, tool arguments, and API responses.

In [ ]:
class SupportRoute(BaseModel):
    intent: Literal["order_status", "refund", "general"]
    urgency: Literal["low", "normal", "high"]
    order_id: str | None = Field(default=None, pattern=r"^(ORD-\d+)?$")
    reason: str = Field(min_length=3, max_length=160)

structured = llm.with_structured_output(SupportRoute)

class RouteState(TypedDict):
    request: str
    route: dict
    destination: str

def classify(state):
    parsed = structured.invoke([
        SystemMessage(content="Classify support requests. Use order_status for tracking, refund for money back, otherwise general."),
        HumanMessage(content=state["request"]),
    ])
    return {"route": parsed.model_dump() if isinstance(parsed, BaseModel) else dict(parsed)}

def route(state):
    return {"refund": "billing", "order_status": "orders", "general": "general"}[state["route"]["intent"]]

routes = StateGraph(RouteState)
routes.add_node("classify", classify)
routes.add_node("orders", lambda s: {"destination": "orders specialist"})
routes.add_node("billing", lambda s: {"destination": "billing specialist"})
routes.add_node("general", lambda s: {"destination": "general support"})
routes.add_edge(START, "classify")
routes.add_conditional_edges("classify", route)
for n in ("orders", "billing", "general"):
    routes.add_edge(n, END)
print(routes.compile().invoke({"request": "Refund order ORD-100 because it arrived damaged", "route": {}, "destination": ""}))
try:
    SupportRoute(intent="refund", urgency="urgent", reason="x")
except ValidationError as exc:
    print("expected validation failure:", exc.errors()[0]["msg"])

In [ ]:
def validate_or_repair(candidate: dict):
    try:
        return SupportRoute.model_validate(candidate)
    except ValidationError:
        print("schema failure caught; repair route activated")
        repaired = dict(candidate)
        repaired["urgency"] = repaired.get("urgency") if repaired.get("urgency") in {"low", "normal", "high"} else "normal"
        repaired["reason"] = repaired.get("reason") if len(repaired.get("reason", "")) >= 3 else "Needs support review"
        return SupportRoute.model_validate(repaired)

print("repaired structured result:", validate_or_repair({"intent": "refund", "urgency": "urgent", "reason": "x"}))

## 5. Advanced checkpointing

**Goal:** inspect history, restart durable state, and branch from a checkpoint.


A checkpointer saves state at super-step boundaries under a thread_id.
get_state reads the latest checkpoint; get_state_history supports replay and
debugging; a new thread can be used for a controlled branch.

~~~mermaid
flowchart LR
 T[thread customer-42] --> C1[checkpoint 1] --> C2[checkpoint 2] --> C3[checkpoint 3]
 C2 -. branch .-> B[alternate future]
~~~

InMemorySaver is appropriate for teaching and tests. The example also
reopens a file-backed SqliteSaver to prove state survives a saver restart, and
uses a historical checkpoint configuration to create a real alternate branch.
Checkpoint state is short-term execution memory; a long-term store is a
separate durable user-fact layer.

In [ ]:
class CounterState(TypedDict):
    counter: int
    events: Annotated[list[str], operator.add]

def count_step(state):
    return {"counter": state["counter"] + 1, "events": ["counter=%d" % (state["counter"] + 1)]}

memory = InMemorySaver()
counter_graph = StateGraph(CounterState)
counter_graph.add_node("count", count_step)
counter_graph.add_edge(START, "count")
counter_graph.add_edge("count", END)
counter_app = counter_graph.compile(checkpointer=memory)
thread = {"configurable": {"thread_id": "customer-42"}}
counter_app.invoke({"counter": 0, "events": []}, thread)
counter_app.invoke({"events": []}, thread)
latest = counter_app.get_state(thread)
history = list(counter_app.get_state_history(thread))
print("latest:", latest.values)
print("history entries:", len(history))
print("checkpoint config:", latest.config["configurable"])

In [ ]:
from langgraph.checkpoint.sqlite import SqliteSaver

db_path = tempfile.NamedTemporaryFile(suffix=".sqlite", delete=False).name
with SqliteSaver.from_conn_string(db_path) as sqlite_saver:
    durable_app = counter_graph.compile(checkpointer=sqlite_saver)
    durable_thread = {"configurable": {"thread_id": "durable-customer-42"}}
    durable_app.invoke({"counter": 0, "events": []}, durable_thread)
    durable_app.invoke({"events": []}, durable_thread)
    print("before saver restart:", durable_app.get_state(durable_thread).values)

with SqliteSaver.from_conn_string(db_path) as reopened_saver:
    reopened_app = counter_graph.compile(checkpointer=reopened_saver)
    print("after saver restart:", reopened_app.get_state(durable_thread).values)

prior = next(item for item in history if item.values.get("counter") == 1)
branch_config = counter_app.update_state(
    prior.config, {"counter": 50, "events": ["historical branch"]}, as_node="count"
)
branch_result = counter_app.invoke({"events": []}, branch_config)
print("historical branch:", branch_result)

from langgraph.store.memory import InMemoryStore
store = InMemoryStore()
store.put(("users", "customer-42"), "preferences", {"language": "English"})
print("long-term store fact:", store.get(("users", "customer-42"), "preferences").value)
Path(db_path).unlink(missing_ok=True)

## 6. Human-in-the-loop agent control

**Goal:** pause before a side effect and exercise approve, edit, reject, and escalation paths.


interrupt persists graph state and returns a review payload. A later
Command(resume=...) continues the same thread. The human can approve, edit,
reject, or trigger escalation with Command(goto=...). The approved path invokes
the refund tool after validating the edited amount, so approval sits directly
before the side effect.

~~~mermaid
flowchart TD
 A[Agent proposes tool call] --> H[interrupt: action and args]
 H -->|approve/edit| T[execute]
 H -->|reject| R[revise response]
 H -->|risk/failures| E[escalation]
~~~

Use approval for spending, irreversible actions, sensitive access, and external
communication. Validate every resume value.

In [ ]:
class ApprovalState(TypedDict):
    action: str
    amount: float
    approved: bool
    outcome: str

def propose(s):
    return {"action": "issue_refund", "amount": 80.0}

def review(s):
    decision = interrupt({"action": s["action"], "amount": s["amount"],
                           "choices": ["approve", "edit", "reject"]})
    if decision["choice"] == "approve":
        return Command(goto="execute", update={"approved": True})
    if decision["choice"] == "edit":
        amount = float(decision["amount"])
        if not 0 < amount <= 500:
            return Command(goto="escalate", update={"outcome": "edited amount escalated"})
        return Command(goto="execute", update={"approved": True, "amount": amount})
    return Command(goto="reject", update={"approved": False})

def execute(s):
    tool_result = calculate_refund.invoke({"amount": s["amount"], "percentage": 100})
    return {"outcome": "approved tool result: " + tool_result}
def reject(s): return {"outcome": "refund rejected by reviewer"}
def escalate(s): return {"outcome": "queued for specialist escalation"}

hitl = StateGraph(ApprovalState)
for n, fn in [("propose", propose), ("review", review), ("execute", execute),
              ("reject", reject), ("escalate", escalate)]:
    hitl.add_node(n, fn)
hitl.add_edge(START, "propose")
hitl.add_edge("propose", "review")
for n in ("execute", "reject", "escalate"):
    hitl.add_edge(n, END)
hitl_app = hitl.compile(checkpointer=InMemorySaver())
def run_review(thread_id, decision):
    review_thread = {"configurable": {"thread_id": thread_id}}
    paused = hitl_app.invoke({"action": "", "amount": 0, "approved": False, "outcome": ""}, review_thread)
    print(thread_id, "paused choices:", paused["__interrupt__"][0].value["choices"])
    return hitl_app.invoke(Command(resume=decision), review_thread)["outcome"]

approval_outcomes = {
    "approve": run_review("refund-approve", {"choice": "approve"}),
    "edit": run_review("refund-edit", {"choice": "edit", "amount": 55}),
    "reject": run_review("refund-reject", {"choice": "reject"}),
    "escalate": run_review("refund-escalate", {"choice": "edit", "amount": 900}),
}
print("approval outcomes:", approval_outcomes)
assert "approved tool result" in approval_outcomes["approve"]
assert "approved tool result" in approval_outcomes["edit"]
assert approval_outcomes["reject"] == "refund rejected by reviewer"
assert approval_outcomes["escalate"] == "queued for specialist escalation"
print("Human-in-the-loop path assertions passed")

## 7. Subgraph state boundaries

**Goal:** map private child state to a minimal public parent result.


A compiled graph can be used as a node, but parent and child need not share a
schema. An adapter maps parent inputs into private child state and maps only a
public result back. This prevents internal prompts, scores, or sensitive fields
from leaking into unrelated nodes.

~~~mermaid
flowchart LR
 P[Parent request] --> A[adapter]
 A --> C[private child state]
 C --> A
 A --> R[Parent resolution]
~~~

In [ ]:
class Parent(TypedDict):
    customer_id: str
    request: str
    resolution: str

class BillingPrivate(TypedDict):
    account_id: str
    issue: str
    risk_score: int
    public_result: str

def score(s):
    return {"risk_score": 90 if "charge" in s["issue"].lower() else 20}
def resolve(s):
    return {"public_result": "billing specialist review required" if s["risk_score"] >= 80 else "standard billing explanation"}

child = StateGraph(BillingPrivate)
child.add_node("score", score)
child.add_node("resolve", resolve)
child.add_edge(START, "score")
child.add_edge("score", "resolve")
child.add_edge("resolve", END)
child_app = child.compile(checkpointer=InMemorySaver())

def adapter(s):
    result = child_app.invoke({"account_id": s["customer_id"], "issue": s["request"],
                               "risk_score": 0, "public_result": ""},
                              {"configurable": {"thread_id": "billing-" + s["customer_id"]}})
    return {"resolution": result["public_result"]}

parent = StateGraph(Parent)
parent.add_node("private_billing_subgraph", adapter)
parent.add_edge(START, "private_billing_subgraph")
parent.add_edge("private_billing_subgraph", END)
print(parent.compile().invoke({"customer_id": "C-7", "request": "unknown charge", "resolution": ""}))

## 8. Robust parallel execution

**Goal:** preserve partial failures and stable ordering across dynamic workers.


Parallel I/O reduces latency but introduces partial failure, retries, timeout,
cancellation, and ordering questions. Send creates dynamic workers; a reducer
collects explicit success/failure records. Sort by a stable key at fan-in when
business order matters.

~~~mermaid
flowchart TD
 S[Dispatch] -->|Send N| A[async worker]
 S -->|Send N| B[async worker]
 S -->|Send N| C[async worker]
 A --> J[fan-in: sort and inspect]
 B --> J
 C --> J
~~~

In [ ]:
class ParallelState(TypedDict):
    regions: list[str]
    reports: Annotated[list[dict], operator.add]
    final: list[dict]

class Worker(TypedDict):
    region: str
    index: int
    reports: Annotated[list[dict], operator.add]

async def fetch(region, index):
    await asyncio.sleep(0.03 if region != "APAC" else 0.01)
    if region == "LATAM":
        raise ConnectionError("regional endpoint unavailable")
    return {"region": region, "index": index, "status": "ok", "sales": 120 + index * 10}

async def worker(s):
    for attempt in range(1, 3):
        try:
            report = await asyncio.wait_for(fetch(s["region"], s["index"]), timeout=0.2)
            return {"reports": [report]}
        except (ConnectionError, asyncio.TimeoutError) as exc:
            if attempt == 2:
                return {"reports": [{"region": s["region"], "index": s["index"],
                                     "status": "failed", "error": str(exc)}]}
            await asyncio.sleep(0.01)

def dispatch(s):
    return [Send("worker", {"region": r, "index": i, "reports": []}) for i, r in enumerate(s["regions"])]
def join(s):
    return {"final": sorted(s["reports"], key=lambda x: x["index"])}

parallel = StateGraph(ParallelState)
parallel.add_node("dispatch", lambda s: {})
parallel.add_node("worker", worker)
parallel.add_node("join", join)
parallel.add_edge(START, "dispatch")
parallel.add_conditional_edges("dispatch", dispatch, ["worker"])
parallel.add_edge("worker", "join")
parallel.add_edge("join", END)
parallel_result = await parallel.compile().ainvoke(
    {"regions": ["EMEA", "APAC", "LATAM"], "reports": [], "final": []})
print(parallel_result["final"])

## 9. Multi-agent patterns

**Goal:** separate supervisor routing from specialist reasoning and integration.


Multiple model calls do not automatically make a multi-agent system. Give each
role a narrow responsibility and explicit boundary.

| Pattern | Use | Trade-off |
|---|---|---|
| supervisor | route to specialists | supervisor bottleneck |
| handoff | transfer ownership | state and permissions cross boundary |
| agent-as-subgraph | reusable specialist | boundary mapping required |
| planner-executor | long tasks | plan can become stale |
| critic/debate | quality/risk review | extra latency and correlated errors |

~~~mermaid
flowchart TD
 U[User] --> S[Supervisor]
 S --> B[Billing specialist]
 S --> O[Orders specialist]
 B --> S
 O --> S
 S --> F[Final answer]
~~~

Use isolated memory for specialist-private context and shared memory only for
authorized common facts. A supervisor routes and integrates; it should not
bypass a specialist safety policy.

In [ ]:
class Multi(TypedDict):
    request: str
    plan: list[str]
    notes: Annotated[list[str], operator.add]
    answer: str

def plan(s):
    decision = get_text(llm.invoke("Classify as exactly one word: orders or billing. Request: " + s["request"])).lower()
    return {"plan": ["orders"] if "order" in decision else ["billing"]}
def billing(s):
    note = get_text(llm.invoke("You are a billing specialist. Give one concise next step for: " + s["request"]))
    return {"notes": ["billing specialist: " + note]}
def orders(s):
    note = get_text(llm.invoke("You are an orders specialist. Give one concise next step for: " + s["request"]))
    return {"notes": ["orders specialist: " + note]}
def choose(s): return s["plan"][0]
def integrate(s): return {"answer": " | ".join(s["notes"])}

multi = StateGraph(Multi)
for n, fn in [("plan", plan), ("billing", billing), ("orders", orders), ("integrate", integrate)]:
    multi.add_node(n, fn)
multi.add_edge(START, "plan")
multi.add_conditional_edges("plan", choose)
multi.add_edge("billing", "integrate")
multi.add_edge("orders", "integrate")
multi.add_edge("integrate", END)
multi_result = multi.compile().invoke({"request": "Where is my order?", "plan": [], "notes": [], "answer": ""})
print(multi_result["answer"])
assert multi_result["plan"] == ["orders"]
assert multi_result["notes"] and "orders specialist:" in multi_result["notes"][0]
print("LLM supervisor and specialist assertions passed")
print("Replace each specialist with a compiled private agent subgraph when it needs tools and memory.")

## 10. Runtime context and configuration

**Goal:** keep identity and permissions outside state while filtering tools in code.


State is business data in the graph trajectory. Runtime context is request
scoped data such as user, tenant, permissions, or dependencies. Configurable
settings can carry a model role, tags, or feature flags.

~~~mermaid
flowchart LR
 S[State] --> N[Node]
 C[Runtime context: identity and permissions] --> N
 K[Configurable settings] --> N
 N --> O[Authorized result]
~~~

This separation prevents secrets and request metadata from accidentally entering
checkpoints. Enforce permissions in code, not only in prompts.

In [ ]:
class UserContext(TypedDict):
    user_id: str
    tenant: str
    allowed_tools: list[str]

class RuntimeState(TypedDict):
    request: str
    selected_model: str
    response: str
    tool_calls: list[str]

def runtime_policy(state, runtime: Runtime[UserContext], config: RunnableConfig):
    context = runtime.context
    model_role = config.get("configurable", {}).get("model", "default")
    allowed = [calculate_refund] if "calculate_refund" in context["allowed_tools"] else []
    denied = "refund" in state["request"].lower() and not allowed
    if denied:
        return {"selected_model": model_role, "response": "Denied for user=%s: refund tool is not permitted" % context["user_id"], "tool_calls": []}
    selected = llm.bind_tools(allowed).invoke(state["request"]) if allowed else llm.invoke(state["request"])
    calls = [call["name"] for call in getattr(selected, "tool_calls", [])]
    response = "Allowed for tenant=%s using role=%s; filtered tools=%s; model selected=%s" % (context["tenant"], model_role, [tool.name for tool in allowed], calls)
    return {"selected_model": model_role, "response": response, "tool_calls": calls}

runtime_graph = StateGraph(RuntimeState)
runtime_graph.add_node("policy", runtime_policy)
runtime_graph.add_edge(START, "policy")
runtime_graph.add_edge("policy", END)
runtime_app = runtime_graph.compile()
denied_runtime = runtime_app.invoke(
    {"request": "request a refund", "selected_model": "", "response": "", "tool_calls": []},
    context={"user_id": "u-7", "tenant": "acme", "allowed_tools": []},
    config={"configurable": {"model": "fast"}},
)
allowed_runtime = runtime_app.invoke(
    {"request": "request a refund", "selected_model": "", "response": "", "tool_calls": []},
    context={"user_id": "u-8", "tenant": "acme", "allowed_tools": ["calculate_refund"]},
    config={"configurable": {"model": "fast"}},
)
print("denied runtime:", denied_runtime)
print("allowed runtime:", allowed_runtime)
assert denied_runtime["tool_calls"] == []
assert "calculate_refund" in allowed_runtime["response"]
print("Runtime permission and tool-filter assertions passed")

## Final checklist

Covered: bounded ReAct agents; agent state and stopping; state, token, message,
custom-progress, and event streaming; tool errors, retries, timeouts,
fallbacks, and multiple calls; Pydantic structured output and validation;
checkpoint history and branching; short-term versus long-term memory; human
approval, edit, reject, and escalation; private subgraph schemas; async
dynamic fan-out and stable ordering; supervisor, specialist, planner patterns;
and runtime context, permissions, and configuration.

Durable database savers, tracing, security review, and evaluation are
production layers to add around these patterns.

## Practice checklist

- Add a hard wall-clock budget to the ReAct loop.
- Print only selected fields from streaming events rather than full message metadata.
- Force a timeout and compare retryable versus non-retryable errors.
- Run all human-review outcomes and add a new escalation rule.
- Replace the specialist functions with compiled private subgraphs.
- Add a second tenant and prove that its allowed tools are filtered independently.

### Security boundary

Tool validation, permission checks, and explicit error observations belong in
code. Network results can be unavailable or untrusted and must be treated as
data, not instructions. Never rely on prompts alone for authorization, and
never execute untrusted model-generated code in the application process.